# 02 — Metric Catalog and Benchmark Design

## Purpose

This notebook translates the 2023 SPARCS data-audit findings into formal,
reproducible specifications for hospital operational metrics, category mappings,
peer benchmarks, denominator rules, exclusions, and small-cell suppression.

## Main Method

Metadata-driven analytical design using the validated outputs from
`01_data_audit.ipynb`.

This notebook defines how metrics will be calculated. It does not yet calculate
hospital performance from the raw discharge records.

## Goals

1. Validate that required source fields passed the data audit.
2. Define the analytical grain and aggregation standards.
3. Create a machine-readable metric catalog.
4. Define denominator and exclusion rules.
5. Create categorical mapping specifications.
6. Define descriptive peer-group LOS benchmarks.
7. Distinguish peer benchmarks from future model-predicted LOS.
8. Define conservative small-cell suppression rules.
9. Validate and export the specifications.
10. Generate business-readable metric documentation.

## Out of Scope

- Raw-data cleaning
- Feature engineering
- Final KPI calculation
- Machine-learning model training
- Risk-adjusted cost modeling
- Power BI semantic modeling
- DAX implementation
- Multi-year schema integration
- Readmission measurement
- Occupancy or capacity measurement
- Causal performance attribution

## Analytical Grain

The source-data grain is one inpatient discharge.

Metrics may be aggregated by facility, APR-DRG, severity, payer, geography,
admission type, and other approved dimensions.

## Critical Benchmark Distinction

In this notebook, `peer-expected LOS` means the average LOS among comparable
discharges in a defined statewide peer group.

It is a descriptive benchmark—not a machine-learning prediction, clinical
expectation, risk-adjusted quality measure, or causal estimate.

A later notebook will develop and validate a predictive expected-LOS model.

## 1. Imports

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

## 2. Project Paths

In [2]:
def find_project_root(start_path):
    """Find the repository root using the existing project charter."""

    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        charter_path = candidate / "docs" / "project_charter.md"

        if charter_path.exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Expected docs/project_charter.md "
        "in the current directory or one of its parents."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

AUDIT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "metric_catalog"
DOCS_DIR = PROJECT_ROOT / "docs"

assert AUDIT_DIR.exists(), f"Audit directory not found: {AUDIT_DIR}"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root detected successfully.")
print("Project folder:", PROJECT_ROOT.name)
print("Audit Directory detected successfully.")
print("Audit Directory:", AUDIT_DIR.name)
print("Output Directory detected successfully.")
print("Output Directory:", OUTPUT_DIR.name)

Project root detected successfully.
Project folder: 04_hospital_operations_powerbi
Audit Directory detected successfully.
Audit Directory: data_audit
Output Directory detected successfully.
Output Directory: metric_catalog


### Interpretation

The notebook uses the project charter to locate the repository root. It therefore
does not depend on a user-specific local file path.

## 3. Load Notebook 01 outputs

In [3]:
audit_files = {
    "schema": "schema.csv",
    "field_availability": "required_field_availability.csv",
    "column_profile": "column_profile.csv",
    "categorical_counts": "categorical_counts.csv",
    "numeric_audit": "numeric_audit.csv",
    "los_distribution": "los_distribution.csv",
    "financial_distribution": "financial_distribution.csv",
}

missing_audit_files = [
    file_name
    for file_name in audit_files.values()
    if not (AUDIT_DIR / file_name).exists()
]

assert not missing_audit_files, (
    "Missing Notebook 01 outputs: "
    f"{missing_audit_files}"
)

audit_outputs = {
    name: pd.read_csv(AUDIT_DIR / file_name)
    for name, file_name in audit_files.items()
}

schema = audit_outputs["schema"]
field_availability = audit_outputs["field_availability"]
column_profile = audit_outputs["column_profile"]
categorical_counts = audit_outputs["categorical_counts"]
numeric_audit = audit_outputs["numeric_audit"]
los_distribution = audit_outputs["los_distribution"]
financial_distribution = audit_outputs["financial_distribution"]

available_columns = set(schema["column_name"])

print(f"Validated source fields: {len(available_columns)}")
display(field_availability)

Validated source fields: 33


,business_concept,expected_source_field,available
0,Facility identifier,Permanent Facility Id,True
1,Operating certificate,Operating Certificate Number,True
2,Facility name,Facility Name,True
3,Hospital service area,Hospital Service Area,True
4,Hospital county,Hospital County,True
5,Length of stay,Length of Stay,True
6,Admission type,Type of Admission,True
7,Patient disposition,Patient Disposition,True
8,Discharge year,Discharge Year,True
9,APR-DRG code,APR DRG Code,True


## 4. Validate Benchmark-Critical Source Fields

In [4]:
benchmark_required_fields = {
    "Permanent Facility Id",
    "Facility Name",
    "Length of Stay",
    "Discharge Year",
    "APR DRG Code",
    "APR DRG Description",
    "APR Severity of Illness Code",
    "APR Severity of Illness Description",
    "APR Risk of Mortality",
    "Type of Admission",
    "Patient Disposition",
    "Emergency Department Indicator",
    "Payment Typology 1",
    "Total Charges",
    "Total Costs",
}

benchmark_field_validation = pd.DataFrame(
    [
        {
            "source_field": field,
            "available": field in available_columns,
        }
        for field in sorted(benchmark_required_fields)
    ]
)

missing_benchmark_fields = benchmark_field_validation.loc[
    ~benchmark_field_validation["available"],
    "source_field",
].tolist()

display(benchmark_field_validation)

assert not missing_benchmark_fields, (
    "Required benchmark fields are missing: "
    f"{missing_benchmark_fields}"
)

,source_field,available
0,APR DRG Code,True
1,APR DRG Description,True
2,APR Risk of Mortality,True
3,APR Severity of Illness Code,True
4,APR Severity of Illness Description,True
5,Discharge Year,True
6,Emergency Department Indicator,True
7,Facility Name,True
8,Length of Stay,True
9,Patient Disposition,True


### Interpretation

All fields required for the initial descriptive metric and benchmark design were
confirmed by Notebook 01.

This validation establishes field availability only. It does not establish that
every field is analytically valid for every purpose. For example, patient
disposition is available but may represent post-outcome information and will not
be used as a predictor in the primary expected-LOS model.

## 5. Analytical Design Standards

In [5]:
design_standards = pd.DataFrame(
    [
        {
            "standard_id": "SOURCE_GRAIN",
            "decision": "One row represents one inpatient discharge.",
            "rationale": (
                "The public file does not support unique-patient tracking."
            ),
        },
        {
            "standard_id": "TIME_GRAIN",
            "decision": "Initial reporting is annual.",
            "rationale": (
                "Only discharge year is available in the audited file."
            ),
        },
        {
            "standard_id": "LOS_TOP_CODE",
            "decision": (
                "Values recorded as 120+ are represented as 120 only when "
                "a numeric lower bound is required."
            ),
            "rationale": (
                "The exact LOS above the threshold is unknown."
            ),
        },
        {
            "standard_id": "LOS_LABEL",
            "decision": (
                "LOS totals and means using top-coded observations must be "
                "labeled lower-bound estimates."
            ),
            "rationale": (
                "Treating 120+ as exactly 120 understates utilization."
            ),
        },
        {
            "standard_id": "FINANCIAL_VALIDITY",
            "decision": (
                "Missing, nonnumeric, and nonpositive financial values are "
                "excluded from per-discharge financial denominators."
            ),
            "rationale": (
                "Invalid financial records should not distort averages."
            ),
        },
        {
            "standard_id": "RATIO_AGGREGATION",
            "decision": "Ratios use ratio-of-sums.",
            "rationale": (
                "A mean of row-level ratios creates inappropriate weighting."
            ),
        },
        {
            "standard_id": "ROBUST_STATISTICS",
            "decision": (
                "Means are reported with medians and upper percentiles."
            ),
            "rationale": (
                "LOS, charges, and estimated costs are right-skewed."
            ),
        },
        {
            "standard_id": "PEER_BASELINE",
            "decision": (
                "Primary descriptive peers use APR-DRG and severity."
            ),
            "rationale": (
                "These fields provide an initial observable case-complexity "
                "comparison."
            ),
        },
        {
            "standard_id": "FACILITY_EXCLUSION",
            "decision": (
                "A facility is excluded from the peer mean used to benchmark "
                "that facility."
            ),
            "rationale": (
                "A facility should not materially influence its own benchmark."
            ),
        },
        {
            "standard_id": "PRIVACY",
            "decision": (
                "Visible cells with counts from 1 through 10 are suppressed."
            ),
            "rationale": (
                "This is a conservative project reporting standard."
            ),
        },
    ]
)

design_standards

,standard_id,decision,rationale
0,SOURCE_GRAIN,One row represents one inpatient discharge.,The public file does not support unique-patient tracking.
1,TIME_GRAIN,Initial reporting is annual.,Only discharge year is available in the audited file.
2,LOS_TOP_CODE,Values recorded as 120+ are represented as 120 only when a numeric lower bound is required.,The exact LOS above the threshold is unknown.
3,LOS_LABEL,LOS totals and means using top-coded observations must be labeled lower-bound estimates.,Treating 120+ as exactly 120 understates utilization.
4,FINANCIAL_VALIDITY,"Missing, nonnumeric, and nonpositive financial values are excluded from per-discharge financial denominators.",Invalid financial records should not distort averages.
5,RATIO_AGGREGATION,Ratios use ratio-of-sums.,A mean of row-level ratios creates inappropriate weighting.
6,ROBUST_STATISTICS,Means are reported with medians and upper percentiles.,"LOS, charges, and estimated costs are right-skewed."
7,PEER_BASELINE,Primary descriptive peers use APR-DRG and severity.,These fields provide an initial observable case-complexity comparison.
8,FACILITY_EXCLUSION,A facility is excluded from the peer mean used to benchmark that facility.,A facility should not materially influence its own benchmark.
9,PRIVACY,Visible cells with counts from 1 through 10 are suppressed.,This is a conservative project reporting standard.


## 6. Metric-Catalog Structure

In [6]:
metric_columns = [
    "metric_id",
    "metric_name",
    "metric_group",
    "business_definition",
    "calculation_rule",
    "numerator",
    "denominator",
    "required_fields",
    "valid_record_rule",
    "aggregation_method",
    "display_format",
    "direction",
    "suppression_rule_id",
    "important_caveat",
    "status",
]


def define_metric(
    metric_id,
    metric_name,
    metric_group,
    business_definition,
    calculation_rule,
    numerator,
    denominator,
    required_fields,
    valid_record_rule,
    aggregation_method,
    display_format,
    direction,
    suppression_rule_id,
    important_caveat,
    status="APPROVED",
):
    return {
        "metric_id": metric_id,
        "metric_name": metric_name,
        "metric_group": metric_group,
        "business_definition": business_definition,
        "calculation_rule": calculation_rule,
        "numerator": numerator,
        "denominator": denominator,
        "required_fields": required_fields,
        "valid_record_rule": valid_record_rule,
        "aggregation_method": aggregation_method,
        "display_format": display_format,
        "direction": direction,
        "suppression_rule_id": suppression_rule_id,
        "important_caveat": important_caveat,
        "status": status,
    }

## 7. Base-Volume Metrics

In [7]:
base_metrics = [
    define_metric(
        metric_id="discharge_count",
        metric_name="Discharges",
        metric_group="Volume",
        business_definition=(
            "Number of inpatient discharge records in the selected context."
        ),
        calculation_rule="COUNT_ROWS",
        numerator="Number of eligible discharge rows",
        denominator="Not applicable",
        required_fields="",
        valid_record_rule="Include all released discharge records",
        aggregation_method="Count",
        display_format="#,0",
        direction="Context",
        suppression_rule_id="COUNT_1_TO_10",
        important_caveat=(
            "Discharge count is not a unique-patient count."
        ),
    ),
    define_metric(
        metric_id="facility_count",
        metric_name="Facilities",
        metric_group="Volume",
        business_definition=(
            "Number of distinct nonmissing permanent facility identifiers."
        ),
        calculation_rule="DISTINCT_COUNT(Permanent Facility Id)",
        numerator="Distinct valid facility identifiers",
        denominator="Not applicable",
        required_fields="Permanent Facility Id",
        valid_record_rule="Permanent Facility Id is nonmissing",
        aggregation_method="Distinct count",
        display_format="#,0",
        direction="Context",
        suppression_rule_id="NONE",
        important_caveat=(
            "Facility identifiers may be suppressed for some records."
        ),
    ),
]

## 8. LOS Metrics

In [8]:
los_metrics = [
    define_metric(
        metric_id="total_los_days_lower_bound",
        metric_name="Total LOS Days — Lower Bound",
        metric_group="Length of Stay",
        business_definition=(
            "Total observed inpatient days after representing 120+ as 120."
        ),
        calculation_rule="SUM(LOS numeric lower bound)",
        numerator="Sum of valid numeric LOS lower bounds",
        denominator="Not applicable",
        required_fields="Length of Stay",
        valid_record_rule=(
            "LOS is numeric or is the recognized 120+ top-coded value"
        ),
        aggregation_method="Sum",
        display_format="#,0",
        direction="Context; volume-dependent",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "This understates exact inpatient days because 120+ values "
            "are right-censored."
        ),
    ),
    define_metric(
        metric_id="average_los_lower_bound",
        metric_name="Average LOS — Lower Bound",
        metric_group="Length of Stay",
        business_definition=(
            "Average numeric LOS after representing 120+ as 120."
        ),
        calculation_rule=(
            "SUM(LOS numeric lower bound) / COUNT(valid LOS records)"
        ),
        numerator="Total LOS days lower bound",
        denominator="Discharges with valid LOS",
        required_fields="Length of Stay",
        valid_record_rule=(
            "LOS is numeric or is the recognized 120+ top-coded value"
        ),
        aggregation_method="Ratio of sums",
        display_format="0.0",
        direction="Higher values may warrant investigation",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "The result is a lower-bound mean and is sensitive to skewness."
        ),
    ),
    define_metric(
        metric_id="median_los_lower_bound",
        metric_name="Median LOS",
        metric_group="Length of Stay",
        business_definition=(
            "Median LOS after representing 120+ as its observable lower bound."
        ),
        calculation_rule="MEDIAN(LOS numeric lower bound)",
        numerator="Not applicable",
        denominator="Discharges with valid LOS",
        required_fields="Length of Stay",
        valid_record_rule=(
            "LOS is numeric or is the recognized 120+ top-coded value"
        ),
        aggregation_method="Median",
        display_format="0.0",
        direction="Higher values may warrant investigation",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Median LOS is robust but does not replace total-days analysis."
        ),
    ),

    define_metric(
    metric_id="los_interquartile_range",
    metric_name="LOS Interquartile Range",
    metric_group="Length of Stay",
    business_definition=(
        "Difference between the 75th and 25th percentiles of "
        "observable LOS."
    ),
    calculation_rule=(
        "PERCENTILE_CONT(LOS numeric lower bound, 0.75) - "
        "PERCENTILE_CONT(LOS numeric lower bound, 0.25)"
    ),
    numerator="Not applicable",
    denominator="Discharges with valid LOS",
    required_fields="Length of Stay",
    valid_record_rule=(
        "LOS is numeric or is the recognized 120+ top-coded value"
    ),
    aggregation_method="Interquartile range",
    display_format="0.0",
    direction="Higher values indicate greater LOS dispersion",
    suppression_rule_id="DENOMINATOR_LT_11",
    important_caveat=(
        "This describes the middle 50% of stays and should be "
        "interpreted alongside the median and total LOS."
    ),
    ),

    define_metric(
        metric_id="p95_los_lower_bound",
        metric_name="P95 LOS",
        metric_group="Length of Stay",
        business_definition=(
            "Ninety-fifth percentile of the observable LOS lower bound."
        ),
        calculation_rule="PERCENTILE_CONT(LOS numeric lower bound, 0.95)",
        numerator="Not applicable",
        denominator="Discharges with valid LOS",
        required_fields="Length of Stay",
        valid_record_rule=(
            "LOS is numeric or is the recognized 120+ top-coded value"
        ),
        aggregation_method="Percentile",
        display_format="0.0",
        direction="Higher values may warrant investigation",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "The percentile may be affected when top-coded cases are included."
        ),
    ),
    define_metric(
        metric_id="top_coded_los_count",
        metric_name="Top-Coded LOS Discharges",
        metric_group="Length of Stay",
        business_definition=(
            "Number of discharges whose released LOS is 120 days or more."
        ),
        calculation_rule="COUNT(Length of Stay classified as 120+)",
        numerator="Top-coded LOS discharges",
        denominator="Not applicable",
        required_fields="Length of Stay",
        valid_record_rule="Length of Stay matches the confirmed top-code",
        aggregation_method="Count",
        display_format="#,0",
        direction="Context",
        suppression_rule_id="COUNT_1_TO_10",
        important_caveat=(
            "The exact stay duration is unavailable."
        ),
    ),
    define_metric(
        metric_id="top_coded_los_rate",
        metric_name="Top-Coded LOS Rate",
        metric_group="Length of Stay",
        business_definition=(
            "Percentage of valid LOS records that are top-coded."
        ),
        calculation_rule=(
            "Top-coded LOS discharges / discharges with valid LOS"
        ),
        numerator="Top-coded LOS discharges",
        denominator="Discharges with valid LOS",
        required_fields="Length of Stay",
        valid_record_rule="Use records with valid released LOS",
        aggregation_method="Ratio of sums",
        display_format="0.0%",
        direction="Context",
        suppression_rule_id="RATE_SMALL_CELL",
        important_caveat=(
            "This measures censoring prevalence, not clinical performance."
        ),
    ),

        define_metric(
        metric_id="extended_stay_rate",
        metric_name="Extended-Stay Rate",
        metric_group="Length of Stay",
        business_definition=(
            "Percentage of eligible discharges exceeding the approved "
            "extended-stay threshold."
        ),
        calculation_rule=(
            "Discharges exceeding approved LOS threshold / "
            "discharges with valid LOS"
        ),
        numerator="Discharges exceeding the approved LOS threshold",
        denominator="Discharges with valid LOS",
        required_fields="Length of Stay",
        valid_record_rule=(
            "Valid LOS and documented extended-stay threshold"
        ),
        aggregation_method="Ratio of sums",
        display_format="0.0%",
        direction="Lower may indicate better flow, subject to case mix",
        suppression_rule_id="RATE_SMALL_CELL",
        important_caveat=(
            "The threshold must be documented before operational use."
        ),
        status="PENDING_THRESHOLD",
)
]

## 9. Financial Metrics

In [9]:
financial_metrics = [
    define_metric(
        metric_id="total_charges",
        metric_name="Total Charges",
        metric_group="Financial",
        business_definition="Total valid billed hospital charges.",
        calculation_rule="SUM(valid Total Charges)",
        numerator="Sum of valid positive Total Charges",
        denominator="Not applicable",
        required_fields="Total Charges",
        valid_record_rule="Total Charges is numeric and greater than zero",
        aggregation_method="Sum",
        display_format="$#,0",
        direction="Context",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Charges are not reimbursement, revenue, or audited expense."
        ),
    ),
    define_metric(
        metric_id="average_charge_per_discharge",
        metric_name="Average Charge per Discharge",
        metric_group="Financial",
        business_definition=(
            "Average billed charge among discharges with valid charges."
        ),
        calculation_rule=(
            "SUM(valid Total Charges) / COUNT(valid charge records)"
        ),
        numerator="Total valid charges",
        denominator="Discharges with valid positive charges",
        required_fields="Total Charges",
        valid_record_rule="Total Charges is numeric and greater than zero",
        aggregation_method="Ratio of sums",
        display_format="$#,0",
        direction="Context",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat="Highly sensitive to right-skewed observations.",
    ),
    define_metric(
        metric_id="median_charge_per_discharge",
        metric_name="Median Charge per Discharge",
        metric_group="Financial",
        business_definition="Median valid billed charge per discharge.",
        calculation_rule="MEDIAN(valid Total Charges)",
        numerator="Not applicable",
        denominator="Discharges with valid positive charges",
        required_fields="Total Charges",
        valid_record_rule="Total Charges is numeric and greater than zero",
        aggregation_method="Median",
        display_format="$#,0",
        direction="Context",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat="Charges do not measure hospital profitability.",
    ),
    define_metric(
        metric_id="total_estimated_costs",
        metric_name="Total Estimated Costs",
        metric_group="Financial",
        business_definition=(
            "Total released analytical cost estimate."
        ),
        calculation_rule="SUM(valid Total Costs)",
        numerator="Sum of valid positive Total Costs",
        denominator="Not applicable",
        required_fields="Total Costs",
        valid_record_rule="Total Costs is numeric and greater than zero",
        aggregation_method="Sum",
        display_format="$#,0",
        direction="Context",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Total Costs does not represent audited hospital expense."
        ),
    ),
    define_metric(
        metric_id="average_estimated_cost_per_discharge",
        metric_name="Average Estimated Cost per Discharge",
        metric_group="Financial",
        business_definition=(
            "Average estimated cost among records with valid cost values."
        ),
        calculation_rule=(
            "SUM(valid Total Costs) / COUNT(valid cost records)"
        ),
        numerator="Total valid estimated costs",
        denominator="Discharges with valid positive costs",
        required_fields="Total Costs",
        valid_record_rule="Total Costs is numeric and greater than zero",
        aggregation_method="Ratio of sums",
        display_format="$#,0",
        direction="Lower is generally favorable",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Case-mix-adjusted cost performance is not established here."
        ),
    ),
    define_metric(
        metric_id="median_estimated_cost_per_discharge",
        metric_name="Median Estimated Cost per Discharge",
        metric_group="Financial",
        business_definition="Median valid estimated cost per discharge.",
        calculation_rule="MEDIAN(valid Total Costs)",
        numerator="Not applicable",
        denominator="Discharges with valid positive costs",
        required_fields="Total Costs",
        valid_record_rule="Total Costs is numeric and greater than zero",
        aggregation_method="Median",
        display_format="$#,0",
        direction="Lower is generally favorable",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "This is an analytical estimate—not audited expense."
        ),
    ),
    define_metric(
        metric_id="charge_to_cost_ratio",
        metric_name="Charge-to-Cost Ratio",
        metric_group="Financial",
        business_definition=(
            "Aggregate charges divided by aggregate estimated costs."
        ),
        calculation_rule=(
            "SUM(Total Charges for paired-valid records) / "
            "SUM(Total Costs for the same paired-valid records)"
        ),
        numerator="Total Charges among paired-valid records",
        denominator="Total Costs among the same paired-valid records",
        required_fields="Total Charges|Total Costs",
        valid_record_rule=(
            "Both Total Charges and Total Costs are numeric and positive"
        ),
        aggregation_method="Ratio of sums",
        display_format="0.00",
        direction="Context",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Do not calculate the mean of discharge-level charge-to-cost ratios."
        ),
    ),
        define_metric(
        metric_id="estimated_cost_per_inpatient_day",
        metric_name="Estimated Cost per Inpatient Day",
        metric_group="Financial",
        business_definition=(
            "Estimated inpatient cost divided by observable inpatient days."
        ),
        calculation_rule=(
            "SUM(valid estimated costs) / "
            "SUM(valid LOS lower-bound days)"
        ),
        numerator="Total valid estimated costs",
        denominator="Total valid LOS lower-bound days",
        required_fields="Total Costs|Length of Stay",
        valid_record_rule=(
            "Positive estimated cost and valid positive LOS"
        ),
        aggregation_method="Ratio of sums",
        display_format="$#,0",
        direction="Higher values may warrant investigation",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Top-coded LOS understates inpatient days and may therefore "
            "overstate estimated cost per day."
        ),
    )
]

## 10. Admission and Case-Mix Context Metrics

In [10]:
context_metrics = [
    define_metric(
        metric_id="ed_origin_discharge_rate",
        metric_name="ED-Origin Discharge Rate",
        metric_group="Admission Context",
        business_definition=(
            "Percentage of eligible inpatient discharges associated with "
            "the emergency department indicator."
        ),
        calculation_rule=(
            "ED-indicator discharges / discharges with known ED indicator"
        ),
        numerator="Discharges mapped to ED = Yes",
        denominator="Discharges with known ED indicator",
        required_fields="Emergency Department Indicator",
        valid_record_rule="ED indicator is mapped and nonmissing",
        aggregation_method="Ratio of sums",
        display_format="0.0%",
        direction="Context",
        suppression_rule_id="RATE_SMALL_CELL",
        important_caveat=(
            "This should not automatically be described as avoidable ED use."
        ),
    ),
    define_metric(
        metric_id="emergency_admission_rate",
        metric_name="Emergency Admission Rate",
        metric_group="Admission Context",
        business_definition=(
            "Percentage of eligible discharges classified as emergency "
            "admissions."
        ),
        calculation_rule=(
            "Emergency admissions / discharges with known admission type"
        ),
        numerator="Discharges mapped to Emergency",
        denominator="Discharges with known admission type",
        required_fields="Type of Admission",
        valid_record_rule="Admission type is mapped and nonmissing",
        aggregation_method="Ratio of sums",
        display_format="0.0%",
        direction="Context",
        suppression_rule_id="RATE_SMALL_CELL",
        important_caveat=(
            "Admission mix reflects patient population and service structure."
        ),
    ),
    define_metric(
        metric_id="major_extreme_severity_rate",
        metric_name="Major or Extreme Severity Rate",
        metric_group="Case Mix",
        business_definition=(
            "Percentage of discharges classified as major or extreme "
            "APR severity."
        ),
        calculation_rule=(
            "Major-or-extreme severity discharges / discharges with "
            "known severity"
        ),
        numerator="Major or extreme APR-severity discharges",
        denominator="Discharges with known APR severity",
        required_fields="APR Severity of Illness Description",
        valid_record_rule="APR severity is known and mapped",
        aggregation_method="Ratio of sums",
        display_format="0.0%",
        direction="Context",
        suppression_rule_id="RATE_SMALL_CELL",
        important_caveat=(
            "This is a case-mix descriptor—not an outcome metric."
        ),
    ),
    define_metric(
        metric_id="major_extreme_mortality_risk_rate",
        metric_name="Major or Extreme Mortality-Risk Rate",
        metric_group="Case Mix",
        business_definition=(
            "Percentage of discharges classified as major or extreme "
            "APR risk of mortality."
        ),
        calculation_rule=(
            "Major-or-extreme risk discharges / discharges with known "
            "APR mortality risk"
        ),
        numerator="Major or extreme APR mortality-risk discharges",
        denominator="Discharges with known APR mortality risk",
        required_fields="APR Risk of Mortality",
        valid_record_rule="APR mortality-risk category is known and mapped",
        aggregation_method="Ratio of sums",
        display_format="0.0%",
        direction="Context",
        suppression_rule_id="RATE_SMALL_CELL",
        important_caveat=(
            "APR risk of mortality is not observed mortality."
        ),
        ),
        define_metric(
        metric_id="home_discharge_rate",
        metric_name="Home Discharge Rate",
        metric_group="Outcomes",
        business_definition=(
            "Percentage of eligible discharges ending in home or self care."
        ),
        calculation_rule=(
            "Home-or-self-care discharges / "
            "discharges with known eligible disposition"
        ),
        numerator="Discharges to Home or Self Care",
        denominator="Discharges with known eligible disposition",
        required_fields="Patient Disposition",
        valid_record_rule=(
            "Patient disposition is known and mapped"
        ),
        aggregation_method="Ratio of sums",
        display_format="0.0%",
        direction="Context-dependent",
        suppression_rule_id="RATE_SMALL_CELL",
        important_caveat=(
            "The measure is influenced by severity, transfers, post-acute "
            "availability, and patient circumstances."
        ),
    ),

        define_metric(
        metric_id="in_hospital_mortality_rate",
        metric_name="In-Hospital Mortality Rate",
        metric_group="Outcomes",
        business_definition=(
            "Percentage of eligible inpatient discharges recorded as expired."
        ),
        calculation_rule=(
            "Expired discharges / discharges with known eligible disposition"
        ),
        numerator="Discharges recorded as Expired",
        denominator="Discharges with known eligible disposition",
        required_fields="Patient Disposition",
        valid_record_rule=(
            "Patient disposition is known and mapped"
        ),
        aggregation_method="Ratio of sums",
        display_format="0.0%",
        direction="Lower is generally favorable after appropriate context",
        suppression_rule_id="RATE_SMALL_CELL",
        important_caveat=(
            "Raw mortality must not be used as a hospital-quality ranking "
            "without severity and case-mix context."
        ),

    ),
        define_metric(
        metric_id="average_apr_severity_index",
        metric_name="Average APR Severity Index",
        metric_group="Case Mix",
        business_definition=(
            "Average APR severity-of-illness code among discharges with "
            "known severity."
        ),
        calculation_rule=(
            "SUM(APR severity code) / discharges with known APR severity"
        ),
        numerator="Sum of valid APR severity codes",
        denominator="Discharges with known APR severity",
        required_fields="APR Severity of Illness Code",
        valid_record_rule=(
            "APR severity code is valid and known"
        ),
        aggregation_method="Ratio of sums",
        display_format="0.00",
        direction="Context",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "APR severity is ordinal; the average is a descriptive case-mix "
            "index rather than a continuous clinical scale."
        ),
    ),define_metric(
    metric_id="disposition_share",
    metric_name="Disposition Share",
    metric_group="Outcomes",
    business_definition=(
        "Percentage of eligible discharges belonging to the selected "
        "disposition category."
    ),
    calculation_rule=(
        "Discharges in selected disposition category / "
        "discharges with known eligible disposition"
    ),
    numerator="Discharges in the selected disposition category",
    denominator="Discharges with known eligible disposition",
    required_fields="Patient Disposition",
    valid_record_rule="Patient Disposition is known and eligible",
    aggregation_method="Ratio of sums",
    display_format="0.0%",
    direction="Context; interpretation depends on disposition category",
    suppression_rule_id="RATE_SMALL_CELL",
    important_caveat=(
        "Disposition is influenced by severity, transfers, post-acute "
        "availability, and patient circumstances."
    ),
),
]

## 11. Peer-Benchmark Metrics

In [11]:
benchmark_metrics = [
    define_metric(
        metric_id="peer_expected_los_days",
        metric_name="Peer-Expected LOS Days",
        metric_group="Benchmarking",
        business_definition=(
            "Expected total LOS based on statewide peer-group mean LOS, "
            "excluding the benchmarked facility."
        ),
        calculation_rule=(
            "SUM(discharge-level leave-one-facility-out peer mean LOS)"
        ),
        numerator="Sum of eligible peer-expected LOS values",
        denominator="Not applicable",
        required_fields=(
            "Permanent Facility Id|APR DRG Code|"
            "APR Severity of Illness Code|Length of Stay"
        ),
        valid_record_rule=(
            "Valid LOS, facility, APR-DRG, severity, and sufficient peers"
        ),
        aggregation_method="Sum",
        display_format="#,0",
        direction="Context",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "This is a descriptive peer baseline—not a predicted clinical LOS."
        ),
    ),
    define_metric(
        metric_id="los_actual_to_peer_expected_ratio",
        metric_name="LOS Actual-to-Peer-Expected Ratio",
        metric_group="Benchmarking",
        business_definition=(
            "Observed LOS days divided by peer-expected LOS days."
        ),
        calculation_rule=(
            "SUM(observed LOS lower bound) / "
            "SUM(peer-expected LOS)"
        ),
        numerator="Observed LOS days lower bound",
        denominator="Peer-expected LOS days",
        required_fields=(
            "Permanent Facility Id|APR DRG Code|"
            "APR Severity of Illness Code|Length of Stay"
        ),
        valid_record_rule="Discharges with valid observed and expected LOS",
        aggregation_method="Ratio of sums",
        display_format="0.00",
        direction= "Values above 1.00 may warrant investigation",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Unmeasured hospital structure and case-mix differences remain."
        ),
    ),
    define_metric(
        metric_id="excess_los_days_lower_bound",
        metric_name="Excess LOS Days — Lower Bound",
        metric_group="Benchmarking",
        business_definition=(
            "Observed LOS days minus peer-expected LOS days."
        ),
        calculation_rule=(
            "SUM(observed LOS lower bound) - "
            "SUM(peer-expected LOS)"
        ),
        numerator="Observed LOS days minus peer-expected LOS days",
        denominator="Not applicable",
        required_fields=(
            "Permanent Facility Id|APR DRG Code|"
            "APR Severity of Illness Code|Length of Stay"
        ),
        valid_record_rule="Discharges with valid observed and expected LOS",
        aggregation_method="Difference of sums",
        display_format="#,0",
        direction="Positive values may warrant investigation",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Positive excess days identify variation—not preventable days."
        ),    
    ),
    define_metric(
    metric_id="peer_expected_estimated_cost",
    metric_name="Peer-Expected Estimated Cost",
    metric_group="Benchmarking",
    business_definition=(
        "Expected aggregate estimated cost based on statewide "
        "leave-one-facility-out peer mean cost."
    ),
    calculation_rule=(
        "SUM(discharge-level leave-one-facility-out peer mean cost)"
    ),
    numerator="Sum of eligible peer-expected cost values",
    denominator="Not applicable",
    required_fields=(
        "Permanent Facility Id|APR DRG Code|"
        "APR Severity of Illness Code|Total Costs"
    ),
    valid_record_rule=(
        "Valid estimated cost, facility, APR-DRG, severity, "
        "and sufficient peers"
    ),
    aggregation_method="Sum",
    display_format="$#,0",
    direction="Context",
    suppression_rule_id="DENOMINATOR_LT_11",
    important_caveat=(
        "Estimated costs are analytical estimates, not audited expenses."
    ),
),
define_metric(
    metric_id="estimated_cost_actual_to_peer_expected_ratio",
    metric_name="Estimated Cost Actual-to-Peer-Expected Ratio",
    metric_group="Benchmarking",
    business_definition=(
        "Observed estimated cost divided by peer-expected estimated cost."
    ),
    calculation_rule=(
        "SUM(observed estimated cost) / "
        "SUM(peer-expected estimated cost)"
    ),
    numerator="Observed valid estimated cost",
    denominator="Peer-expected estimated cost",
    required_fields=(
        "Permanent Facility Id|APR DRG Code|"
        "APR Severity of Illness Code|Total Costs"
    ),
    valid_record_rule=(
        "Discharges with valid observed and expected estimated cost"
    ),
    aggregation_method="Ratio of sums",
    display_format="0.00",
    direction="Values above 1.00 may warrant investigation",
    suppression_rule_id="DENOMINATOR_LT_11",
    important_caveat=(
        "Regional input prices, hospital structure, and unmeasured case mix "
        "may influence the ratio."
    ),
    ),
    define_metric(
        metric_id="excess_estimated_cost",
        metric_name="Excess Estimated Cost",
        metric_group="Benchmarking",
        business_definition=(
            "Observed estimated cost minus peer-expected estimated cost."
        ),
        calculation_rule=(
            "SUM(observed estimated cost) - "
            "SUM(peer-expected estimated cost)"
        ),
        numerator="Observed estimated cost minus peer-expected estimated cost",
        denominator="Not applicable",
        required_fields=(
            "Permanent Facility Id|APR DRG Code|"
            "APR Severity of Illness Code|Total Costs"
        ),
        valid_record_rule=(
            "Discharges with valid observed and expected estimated cost"
        ),
        aggregation_method="Difference of sums",
        display_format="$#,0",
        direction="Positive values may warrant investigation",
        suppression_rule_id="DENOMINATOR_LT_11",
        important_caveat=(
            "Positive excess estimated cost does not prove waste, "
            "inefficiency, or preventability."
        ),
    )
]

In [12]:
metric_catalog = pd.DataFrame(
    base_metrics
    + los_metrics
    + financial_metrics
    + context_metrics
    + benchmark_metrics,
    columns=metric_columns,
)

metric_catalog

,metric_id,metric_name,metric_group,business_definition,calculation_rule,numerator,denominator,required_fields,valid_record_rule,aggregation_method,display_format,direction,suppression_rule_id,important_caveat,status
0,discharge_count,Discharges,Volume,Number of inpatient discharge records in the selected context.,COUNT_ROWS,Number of eligible discharge rows,Not applicable,,Include all released discharge records,Count,"#,0",Context,COUNT_1_TO_10,Discharge count is not a unique-patient count.,APPROVED
1,facility_count,Facilities,Volume,Number of distinct nonmissing permanent facility identifiers.,DISTINCT_COUNT(Permanent Facility Id),Distinct valid facility identifiers,Not applicable,Permanent Facility Id,Permanent Facility Id is nonmissing,Distinct count,"#,0",Context,NONE,Facility identifiers may be suppressed for some records.,APPROVED
2,total_los_days_lower_bound,Total LOS Days — Lower Bound,Length of Stay,Total observed inpatient days after representing 120+ as 120.,SUM(LOS numeric lower bound),Sum of valid numeric LOS lower bounds,Not applicable,Length of Stay,LOS is numeric or is the recognized 120+ top-coded value,Sum,"#,0",Context; volume-dependent,DENOMINATOR_LT_11,This understates exact inpatient days because 120+ values are right-censored.,APPROVED
3,average_los_lower_bound,Average LOS — Lower Bound,Length of Stay,Average numeric LOS after representing 120+ as 120.,SUM(LOS numeric lower bound) / COUNT(valid LOS records),Total LOS days lower bound,Discharges with valid LOS,Length of Stay,LOS is numeric or is the recognized 120+ top-coded value,Ratio of sums,0.0,Higher values may warrant investigation,DENOMINATOR_LT_11,The result is a lower-bound mean and is sensitive to skewness.,APPROVED
4,median_los_lower_bound,Median LOS,Length of Stay,Median LOS after representing 120+ as its observable lower bound.,MEDIAN(LOS numeric lower bound),Not applicable,Discharges with valid LOS,Length of Stay,LOS is numeric or is the recognized 120+ top-coded value,Median,0.0,Higher values may warrant investigation,DENOMINATOR_LT_11,Median LOS is robust but does not replace total-days analysis.,APPROVED
5,los_interquartile_range,LOS Interquartile Range,Length of Stay,Difference between the 75th and 25th percentiles of observable LOS.,"PERCENTILE_CONT(LOS numeric lower bound, 0.75) - PERCENTILE_CONT(LOS numeric lower bound, 0.25)",Not applicable,Discharges with valid LOS,Length of Stay,LOS is numeric or is the recognized 120+ top-coded value,Interquartile range,0.0,Higher values indicate greater LOS dispersion,DENOMINATOR_LT_11,This describes the middle 50% of stays and should be interpreted alongside the median and total LOS.,APPROVED
6,p95_los_lower_bound,P95 LOS,Length of Stay,Ninety-fifth percentile of the observable LOS lower bound.,"PERCENTILE_CONT(LOS numeric lower bound, 0.95)",Not applicable,Discharges with valid LOS,Length of Stay,LOS is numeric or is the recognized 120+ top-coded value,Percentile,0.0,Higher values may warrant investigation,DENOMINATOR_LT_11,The percentile may be affected when top-coded cases are included.,APPROVED
7,top_coded_los_count,Top-Coded LOS Discharges,Length of Stay,Number of discharges whose released LOS is 120 days or more.,COUNT(Length of Stay classified as 120+),Top-coded LOS discharges,Not applicable,Length of Stay,Length of Stay matches the confirmed top-code,Count,"#,0",Context,COUNT_1_TO_10,The exact stay duration is unavailable.,APPROVED
8,top_coded_los_rate,Top-Coded LOS Rate,Length of Stay,Percentage of valid LOS records that are top-coded.,Top-coded LOS discharges / discharges with valid LOS,Top-coded LOS discharges,Discharges with valid LOS,Length of Stay,Use records with valid released LOS,Ratio of sums,0.0%,Context,RATE_SMALL_CELL,"This measures censoring prevalence, not clinical performance.",APPROVED
9,extended_stay_rate,Extended-Stay Rate,Length of Stay,Percentage of eligible discharges exceeding the approved extended-stay threshold.,Discharges exceeding approved LOS threshold / disc

## 12. Category-Mapping Worksheet

In [13]:
mapping_fields = [
    "Age Group",
    "Type of Admission",
    "Patient Disposition",
    "APR Severity of Illness Description",
    "APR Risk of Mortality",
    "Payment Typology 1",
    "Emergency Department Indicator",
]

category_mapping = (
    categorical_counts.loc[
        categorical_counts["column_name"].isin(mapping_fields)
    ]
    .rename(
        columns={
            "column_name": "source_field",
            "category": "source_value",
        }
    )
    .loc[
        :,
        [
            "source_field",
            "source_value",
            "discharges",
            "percentage",
        ],
    ]
    .copy()
)

category_mapping["target_group"] = category_mapping["source_value"]
category_mapping["mapping_action"] = "RETAIN"
category_mapping["mapping_status"] = "REVIEW_REQUIRED"
category_mapping["mapping_rationale"] = ""

category_mapping = category_mapping.sort_values(
    ["source_field", "discharges"],
    ascending=[True, False],
).reset_index(drop=True)

category_mapping

,source_field,source_value,discharges,percentage,target_group,mapping_action,mapping_status,mapping_rationale
0,APR Risk of Mortality,Minor,1073037,50.4779,Minor,RETAIN,REVIEW_REQUIRED,
1,APR Risk of Mortality,Moderate,455080,21.4079,Moderate,RETAIN,REVIEW_REQUIRED,
2,APR Risk of Mortality,Major,399457,18.7913,Major,RETAIN,REVIEW_REQUIRED,
3,APR Risk of Mortality,Extreme,197706,9.3005,Extreme,RETAIN,REVIEW_REQUIRED,
4,APR Risk of Mortality,[MISSING],474,0.0223,[MISSING],RETAIN,REVIEW_REQUIRED,
5,APR Severity of Illness Description,Moderate,785246,36.9396,Moderate,RETAIN,REVIEW_REQUIRED,
6,APR Severity of Illness Description,Minor,627605,29.5239,Minor,RETAIN,REVIEW_REQUIRED,
7,APR Severity of Illness Description,Major,504585,23.7368,Major,RETAIN,REVIEW_REQUIRED,
8,APR Severity of Illness Description,Extreme,207844,9.7774,Extreme,RETAIN,REVIEW_REQUIRED,
9,APR Severity of Illness Description,[MISSING],474,0.0223,[MISSING],RETAIN,REVIEW_REQUIRED,


In [14]:
def review_categories(source_field):
    return category_mapping.loc[
        category_mapping["source_field"] == source_field
    ].reset_index(drop=True)


review_categories("Payment Typology 1")

,source_field,source_value,discharges,percentage,target_group,mapping_action,mapping_status,mapping_rationale
0,Payment Typology 1,Medicare,867570,40.8123,Medicare,RETAIN,REVIEW_REQUIRED,
1,Payment Typology 1,Medicaid,648713,30.5168,Medicaid,RETAIN,REVIEW_REQUIRED,
2,Payment Typology 1,Private Health Insurance,299131,14.0718,Private Health Insurance,RETAIN,REVIEW_REQUIRED,
3,Payment Typology 1,Blue Cross/Blue Shield,207377,9.7555,Blue Cross/Blue Shield,RETAIN,REVIEW_REQUIRED,
4,Payment Typology 1,Self-Pay,29079,1.3679,Self-Pay,RETAIN,REVIEW_REQUIRED,
5,Payment Typology 1,"Managed Care, Unspecified",27919,1.3134,"Managed Care, Unspecified",RETAIN,REVIEW_REQUIRED,
6,Payment Typology 1,Miscellaneous/Other,24073,1.1324,Miscellaneous/Other,RETAIN,REVIEW_REQUIRED,
7,Payment Typology 1,Federal/State/Local/VA,20518,0.9652,Federal/State/Local/VA,RETAIN,REVIEW_REQUIRED,
8,Payment Typology 1,Department of Corrections,1374,0.0646,Department of Corrections,RETAIN,REVIEW_REQUIRED,


### Required Manual Review

Each released category must be assigned one of the following actions:

- `RETAIN`: preserve the source category
- `GROUP`: combine it into a documented broader category
- `MISSING`: classify it as missing or unavailable
- `EXCLUDE`: exclude it from a specifically documented metric

Categories must not be grouped merely to make charts look cleaner.

Payer grouping, disposition grouping, and missing-category treatment require
special review because aggressive grouping can hide materially different
populations.

In [15]:
mapping_rationales = {
    "Age Group": (
        "Retain the released age bands because they already provide "
        "business-readable demographic categories."
    ),
    "Type of Admission": (
        "Retain valid admission categories because each represents a "
        "distinct operational admission pathway."
    ),
    "Patient Disposition": (
        "Retain released disposition detail to avoid hiding materially "
        "different post-discharge settings."
    ),
    "APR Severity of Illness Description": (
        "Retain the released APR severity categories."
    ),
    "APR Risk of Mortality": (
        "Retain the released APR mortality-risk categories."
    ),
    "Payment Typology 1": (
        "Retain released payer categories; no unsupported payer consolidation "
        "is applied."
    ),
    "Emergency Department Indicator": (
        "Retain the released binary Y/N categories."
    ),
}

category_mapping["target_group"] = category_mapping["source_value"]
category_mapping["mapping_action"] = "RETAIN"
category_mapping["mapping_status"] = "APPROVED"
category_mapping["mapping_rationale"] = (
    category_mapping["source_field"].map(mapping_rationales)
)

missing_category_mask = (
    (
        category_mapping["source_field"].isin(
            [
                "APR Severity of Illness Description",
                "APR Risk of Mortality",
            ]
        )
        & category_mapping["source_value"].eq("[MISSING]")
    )
    |
    (
        category_mapping["source_field"].eq("Type of Admission")
        & category_mapping["source_value"].eq("Not Available")
    )
)

category_mapping.loc[
    missing_category_mask,
    "target_group",
] = "Missing / Not Available"

category_mapping.loc[
    missing_category_mask,
    "mapping_action",
] = "MISSING"

category_mapping.loc[
    missing_category_mask,
    "mapping_rationale",
] = (
    "Classified as unavailable and excluded from metrics requiring "
    "a known category."
)

display(category_mapping)

,source_field,source_value,discharges,percentage,target_group,mapping_action,mapping_status,mapping_rationale
0,APR Risk of Mortality,Minor,1073037,50.4779,Minor,RETAIN,APPROVED,Retain the released APR mortality-risk categories.
1,APR Risk of Mortality,Moderate,455080,21.4079,Moderate,RETAIN,APPROVED,Retain the released APR mortality-risk categories.
2,APR Risk of Mortality,Major,399457,18.7913,Major,RETAIN,APPROVED,Retain the released APR mortality-risk categories.
3,APR Risk of Mortality,Extreme,197706,9.3005,Extreme,RETAIN,APPROVED,Retain the released APR mortality-risk categories.
4,APR Risk of Mortality,[MISSING],474,0.0223,Missing / Not Available,MISSING,APPROVED,Classified as unavailable and excluded from metrics requiring a known category.
5,APR Severity of Illness Description,Moderate,785246,36.9396,Moderate,RETAIN,APPROVED,Retain the released APR severity categories.
6,APR Severity of Illness Description,Minor,627605,29.5239,Minor,RETAIN,APPROVED,Retain the released APR severity categories.
7,APR Severity of Illness Description,Major,504585,23.7368,Major,RETAIN,APPROVED,Retain the released APR severity categories.
8,APR Severity of Illness Description,Extreme,207844,9.7774,Extreme,RETAIN,APPROVED,Retain the released APR severity categories.
9,APR Severity of Illness Description,[MISSING],474,0.0223,Missing / Not Available,MISSING,APPROVED,Classified as unavailable and excluded from metrics requiring a known category.


## 13. Benchmark Specification

In [16]:
benchmark_specification = pd.DataFrame(
    [
        {
            "benchmark_id": "LOS_PEER_MEAN_PRIMARY",
            "benchmark_name": "Primary Peer Mean LOS",
            "outcome": "LOS numeric lower bound",
            "primary_peer_keys": (
                "APR DRG Code|APR Severity of Illness Code"
            ),
            "fallback_peer_keys": "APR DRG Code",
            "facility_excluded_from_own_peer": True,
            "minimum_peer_n": 30,
            "peer_statistic": "Arithmetic mean",
            "intended_use": (
                "Descriptive case-mix-contextualized LOS benchmark"
            ),
            "not_intended_for": (
                "Causal attribution or clinical LOS prediction"
            ),
            "status": "APPROVED_BASELINE",
        },
        {
            "benchmark_id": "LOS_PEER_MEDIAN_CONTEXT",
            "benchmark_name": "Peer Median LOS Context",
            "outcome": "LOS numeric lower bound",
            "primary_peer_keys": (
                "APR DRG Code|APR Severity of Illness Code"
            ),
            "fallback_peer_keys": "APR DRG Code",
            "facility_excluded_from_own_peer": True,
            "minimum_peer_n": 30,
            "peer_statistic": "Median",
            "intended_use": (
                "Robust descriptive comparison alongside peer mean"
            ),
            "not_intended_for": (
                "Expected-total calculation or A/E denominator"
            ),
            "status": "DEFERRED_CONTEXT_IMPLEMENTATION",
        },
        {
            "benchmark_id": "LOS_PREDICTIVE_MODEL",
            "benchmark_name": "Model-Predicted Expected LOS",
            "outcome": "LOS modeling target",
            "primary_peer_keys": "To be defined during modeling",
            "fallback_peer_keys": "Not applicable",
            "facility_excluded_from_own_peer": True,
            "minimum_peer_n": pd.NA,
            "peer_statistic": "Validated model prediction",
            "intended_use": (
                "Later case-mix-adjusted predictive benchmarking"
            ),
            "not_intended_for": (
                "Use before model validation and temporal evaluation"
            ),
            "status": "DEFERRED_TO_MODELING",
        },

        {
        "benchmark_id": "COST_PEER_MEAN_PRIMARY",
        "benchmark_name": "Primary Peer Mean Estimated Cost",
        "outcome": "Valid positive estimated cost",
        "primary_peer_keys": (
            "APR DRG Code|APR Severity of Illness Code"
        ),
        "fallback_peer_keys": "APR DRG Code",
        "facility_excluded_from_own_peer": True,
        "minimum_peer_n": 30,
        "peer_statistic": "Arithmetic mean",
        "intended_use": (
            "Descriptive case-mix-contextualized estimated-cost benchmark"
        ),
        "not_intended_for": (
            "Audited expense, reimbursement, profitability, or causal attribution"
        ),
        "status": "APPROVED_BASELINE",
    },
    {
        "benchmark_id": "COST_PEER_MEDIAN_CONTEXT",
        "benchmark_name": "Peer Median Estimated Cost Context",
        "outcome": "Valid positive estimated cost",
        "primary_peer_keys": (
            "APR DRG Code|APR Severity of Illness Code"
        ),
        "fallback_peer_keys": "APR DRG Code",
        "facility_excluded_from_own_peer": True,
        "minimum_peer_n": 30,
        "peer_statistic": "Median",
        "intended_use": (
            "Robust descriptive estimated-cost comparison"
        ),
        "not_intended_for": (
            "Expected-total calculation or A/E denominator"
        ),
        "status": "DEFERRED_CONTEXT_IMPLEMENTATION",
    }
    ]
)

benchmark_specification

,benchmark_id,benchmark_name,outcome,primary_peer_keys,fallback_peer_keys,facility_excluded_from_own_peer,minimum_peer_n,peer_statistic,intended_use,not_intended_for,status
0,LOS_PEER_MEAN_PRIMARY,Primary Peer Mean LOS,LOS numeric lower bound,APR DRG Code|APR Severity of Illness Code,APR DRG Code,True,30,Arithmetic mean,Descriptive case-mix-contextualized LOS benchmark,Causal attribution or clinical LOS prediction,APPROVED_BASELINE
1,LOS_PEER_MEDIAN_CONTEXT,Peer Median LOS Context,LOS numeric lower bound,APR DRG Code|APR Severity of Illness Code,APR DRG Code,True,30,Median,Robust descriptive comparison alongside peer mean,Expected-total calculation or A/E denominator,DEFERRED_CONTEXT_IMPLEMENTATION
2,LOS_PREDICTIVE_MODEL,Model-Predicted Expected LOS,LOS modeling target,To be defined during modeling,Not applicable,True,<NA>,Validated model prediction,Later case-mix-adjusted predictive benchmarking,Use before model validation and temporal evaluation,DEFERRED_TO_MODELING
3,COST_PEER_MEAN_PRIMARY,Primary Peer Mean Estimated Cost,Valid positive estimated cost,APR DRG Code|APR Severity of Illness Code,APR DRG Code,True,30,Arithmetic mean,Descriptive case-mix-contextualized estimated-cost benchmark,"Audited expense, reimbursement, profitability, or causal attribution",APPROVED_BASELINE
4,COST_PEER_MEDIAN_CONTEXT,Peer Median Estimated Cost Context,Valid positive estimated cost,APR DRG Code|APR Severity of Illness Code,APR DRG Code,True,30,Median,Robust descriptive estimated-cost comparison,Expected-total calculation or A/E denominator,DEFERRED_CONTEXT_IMPLEMENTATION


### Interpretation

The primary benchmark matches discharges using APR-DRG and severity. When the
primary peer group contains fewer than 30 eligible comparison discharges, the
design falls back to APR-DRG alone.

The minimum peer size of 30 is an analytical-stability rule. It is separate from
the fewer-than-11 reporting-suppression policy.

A facility is excluded from its own peer calculation. Conceptually:

`peer mean excluding facility =
(statewide peer LOS sum - facility peer LOS sum) /
(statewide peer discharge count - facility peer discharge count)`

The minimum peer count is evaluated after removing the benchmarked facility.
If the APR-DRG-and-severity peer group has fewer than 30 remaining comparison
discharges, the benchmark falls back to APR-DRG alone. If the fallback group
also has fewer than 30 comparison discharges, peer-expected LOS remains missing.

Patient disposition is excluded from the primary benchmark because it may reflect
events occurring during or after the hospital stay. Including it could absorb part
of the operational variation the benchmark is intended to measure.

## 14. Small-Cell Suppression Policy

In [17]:
suppression_policy = pd.DataFrame(
    [
        {
            "suppression_rule_id": "COUNT_1_TO_10",
            "applies_to": "Displayed discharge count",
            "rule": (
                "Suppress displayed discharge counts between 1 and 10. "
                "Display zero when the true count is zero."
            ),
            "zero_allowed": True,
            "implementation_target": "Power BI measure and public exports",
            "limitation": (
                "Complementary suppression may be required when totals allow "
                "a suppressed value to be inferred."
            ),
        },
        {
            "suppression_rule_id": "NONE",
            "applies_to": "Non-patient operational count",
            "rule": (
                "No small-cell suppression is applied based solely on "
                "the number of facilities."
            ),
            "zero_allowed": True,
            "implementation_target": "Power BI measure and public exports",
            "limitation": (
                "This does not override disclosure review for related "
                "patient-level measures."
            ),
            },
        {
            "suppression_rule_id": "DENOMINATOR_LT_11",
            "applies_to": "Continuous summary or aggregate amount",
            "rule": (
                "Suppress when the contributing discharge count is below 11."
            ),
            "zero_allowed": True,
            "implementation_target": "Power BI measure and public exports",
            "limitation": (
                "The threshold protects small cells but does not eliminate "
                "all differencing risk."
            ),
        },
        {
            "suppression_rule_id": "RATE_SMALL_CELL",
            "applies_to": "Rate or percentage",
            "rule": (
                "Suppress when the denominator is below 11, when the numerator "
                "is between 1 and 10, or when the denominator minus numerator "
                "is between 1 and 10."
            ),
            "zero_allowed": True,
            "implementation_target": "Power BI measure and public exports",
            "limitation": (
                "Dashboard totals and cross-filtering require inference testing."
            ),
        },
    ]
)
count_rule_text = suppression_policy.loc[
    suppression_policy["suppression_rule_id"].eq("COUNT_1_TO_10"),
    "rule",
].iloc[0].lower()

suppression_policy

,suppression_rule_id,applies_to,rule,zero_allowed,implementation_target,limitation
0,COUNT_1_TO_10,Displayed discharge count,Suppress displayed discharge counts between 1 and 10. Display zero when the true count is zero.,True,Power BI measure and public exports,Complementary suppression may be required when totals allow a suppressed value to be inferred.
1,NONE,Non-patient operational count,No small-cell suppression is applied based solely on the number of facilities.,True,Power BI measure and public exports,This does not override disclosure review for related patient-level measures.
2,DENOMINATOR_LT_11,Continuous summary or aggregate amount,Suppress when the contributing discharge count is below 11.,True,Power BI measure and public exports,The threshold protects small cells but does not eliminate all differencing risk.
3,RATE_SMALL_CELL,Rate or percentage,"Suppress when the denominator is below 11, when the numerator is between 1 and 10, or when the denominator minus numerator is between 1 and 10.",True,Power BI measure and public exports,Dashboard totals and cross-filtering require inference testing.


## 15. Validate Source-Field References

In [18]:
def parse_required_fields(field_string):
    if pd.isna(field_string) or not str(field_string).strip():
        return set()

    return {
        field.strip()
        for field in str(field_string).split("|")
        if field.strip()
    }


field_validation_rows = []

for _, row in metric_catalog.iterrows():
    required = parse_required_fields(row["required_fields"])
    missing = sorted(required - available_columns)

    field_validation_rows.append(
        {
            "metric_id": row["metric_id"],
            "required_field_count": len(required),
            "missing_fields": "|".join(missing),
            "passed": len(missing) == 0,
        }
    )

metric_field_validation = pd.DataFrame(field_validation_rows)

display(metric_field_validation)

assert metric_field_validation["passed"].all(), (
    "One or more metrics reference unavailable source fields."
)

,metric_id,required_field_count,missing_fields,passed
0,discharge_count,0,,True
1,facility_count,1,,True
2,total_los_days_lower_bound,1,,True
3,average_los_lower_bound,1,,True
4,median_los_lower_bound,1,,True
5,los_interquartile_range,1,,True
6,p95_los_lower_bound,1,,True
7,top_coded_los_count,1,,True
8,top_coded_los_rate,1,,True
9,extended_stay_rate,1,,True


## 16. Catalog-Quality Validation

In [19]:
validation_results = []

validation_results.append(
    {
        "validation_test": "Metric IDs are unique",
        "passed": metric_catalog["metric_id"].is_unique,
        "details": "",
    }
)

validation_results.append(
    {
        "validation_test": "Metric names are populated",
        "passed": metric_catalog["metric_name"].notna().all(),
        "details": "",
    }
)

validation_results.append(
    {
        "validation_test": "Calculation rules are populated",
        "passed": metric_catalog["calculation_rule"].notna().all(),
        "details": "",
    }
)

validation_results.append(
    {
        "validation_test": "Suppression rules are valid",
        "passed": set(
            metric_catalog["suppression_rule_id"]
        ).issubset(
            set(suppression_policy["suppression_rule_id"])
        ),
        "details": "",
    }
)

validation_results.append(
    {
        "validation_test": "All required source fields are available",
        "passed": metric_field_validation["passed"].all(),
        "details": "",
    }
)

unsupported_metric_terms = [
    "readmission rate",
    "occupancy rate",
    "unique patient count",
]

catalog_text = " ".join(
    metric_catalog.astype(str).fillna("").stack().tolist()
).lower()

found_unsupported_terms = [
    term
    for term in unsupported_metric_terms
    if term in catalog_text
]

validation_results.append(
    {
        "validation_test": "No unsupported metric claims",
        "passed": len(found_unsupported_terms) == 0,
        "details": "|".join(found_unsupported_terms),
    }
)

validation_results.append(
    {
        "validation_test": "Small discharge counts are suppressed",
        "passed": (
            "suppress" in count_rule_text
            and "do not suppress" not in count_rule_text
        ),
        "details": count_rule_text,
    }
)

allowed_mapping_actions = {
    "RETAIN",
    "GROUP",
    "MISSING",
    "EXCLUDE",
}

validation_results.extend(
    [
        {
            "validation_test": "Category mappings are unique",
            "passed": ~category_mapping.duplicated(
                ["source_field", "source_value"]
            ).any(),
            "details": "",
        },
        {
            "validation_test": "Mapping actions are valid",
            "passed": set(
                category_mapping["mapping_action"]
            ).issubset(allowed_mapping_actions),
            "details": "",
        },
        {
            "validation_test": "Mapping target groups are populated",
            "passed": category_mapping["target_group"]
            .fillna("")
            .str.strip()
            .ne("")
            .all(),
            "details": "",
        },
        {
            "validation_test": "Mapping rationales are populated",
            "passed": category_mapping["mapping_rationale"]
            .fillna("")
            .str.strip()
            .ne("")
            .all(),
            "details": "",
        },
        {
            "validation_test": "Category mappings are approved",
            "passed": category_mapping["mapping_status"]
            .eq("APPROVED")
            .all(),
            "details": "",
        },
    ]
)

validation_results = pd.DataFrame(validation_results)

validation_results

,validation_test,passed,details
0,Metric IDs are unique,True,
1,Metric names are populated,True,
2,Calculation rules are populated,True,
3,Suppression rules are valid,True,
4,All required source fields are available,True,
5,No unsupported metric claims,True,
6,Small discharge counts are suppressed,True,suppress displayed discharge counts between 1 and 10. display zero when the true count is zero.
7,Category mappings are unique,True,
8,Mapping actions are valid,True,
9,Mapping target groups are populated,True,


In [20]:
assert validation_results["passed"].all(), (
    "The metric specification failed one or more validation tests."
)

unresolved_mappings = (
    category_mapping["mapping_status"] == "REVIEW_REQUIRED"
).sum()

print(f"Unresolved category mappings: {unresolved_mappings}")

assert unresolved_mappings == 0, (
    "Category mappings remain unresolved. Review them before export."
)

Unresolved category mappings: 0


## 17. Export Machine-Readable Specifications

In [21]:
export_tables = {
    "metric_catalog.csv": metric_catalog,
    "category_mapping.csv": category_mapping,
    "benchmark_specification.csv": benchmark_specification,
    "suppression_policy.csv": suppression_policy,
    "design_standards.csv": design_standards,
    "benchmark_field_validation.csv": benchmark_field_validation,
    "metric_field_validation.csv": metric_field_validation,
    "validation_results.csv": validation_results,
}
export_manifest = []

for file_name, dataframe in export_tables.items():
    output_path = OUTPUT_DIR / file_name

    dataframe.to_csv(output_path, index=False)

    assert output_path.exists()
    assert output_path.stat().st_size > 0

    export_manifest.append(
        {
            "file_name": file_name,
            "row_count": len(dataframe),
            "output_path": output_path.relative_to(PROJECT_ROOT).as_posix(),
        }
    )

export_manifest = pd.DataFrame(export_manifest)

manifest_path = OUTPUT_DIR / "export_manifest.csv"
export_manifest.to_csv(manifest_path, index=False)

assert manifest_path.exists()
assert manifest_path.stat().st_size > 0

display(export_manifest)

,file_name,row_count,output_path
0,metric_catalog.csv,32,outputs/metric_catalog/metric_catalog.csv
1,category_mapping.csv,51,outputs/metric_catalog/category_mapping.csv
2,benchmark_specification.csv,5,outputs/metric_catalog/benchmark_specification.csv
3,suppression_policy.csv,4,outputs/metric_catalog/suppression_policy.csv
4,design_standards.csv,10,outputs/metric_catalog/design_standards.csv
5,benchmark_field_validation.csv,15,outputs/metric_catalog/benchmark_field_validation.csv
6,metric_field_validation.csv,32,outputs/metric_catalog/metric_field_validation.csv
7,validation_results.csv,12,outputs/metric_catalog/validation_results.csv


## 18. Generate docs/metric_catalog.md

In [22]:
def dataframe_to_markdown(dataframe, columns):
    selected = dataframe.loc[:, columns].fillna("").astype(str)

    def escape_value(value):
        return value.replace("|", r"\|").replace("\n", " ")

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"

    rows = [
        "| "
        + " | ".join(escape_value(value) for value in row)
        + " |"
        for row in selected.itertuples(index=False, name=None)
    ]

    return "\n".join([header, separator, *rows])


metric_table_markdown = dataframe_to_markdown(
    metric_catalog,
    [
        "metric_name",
        "metric_group",
        "business_definition",
        "calculation_rule",
        "important_caveat",
    ],
)

benchmark_table_markdown = dataframe_to_markdown(
    benchmark_specification,
    [
        "benchmark_name",
        "primary_peer_keys",
        "minimum_peer_n",
        "intended_use",
        "status",
    ],
)

metric_document = f"""# Metric Catalog — Hospital Operations & Cost Efficiency

## Purpose

This document defines the approved metrics, denominator rules, benchmark logic,
and interpretation limitations for the initial 2023 SPARCS analysis.

## Analytical Grain

The source grain is one inpatient discharge. Discharge counts must not be
interpreted as unique-patient counts.

## Aggregation Standards

- Ratios use ratio-of-sums.
- LOS, charges, and estimated costs require robust statistics.
- LOS values recorded as 120+ are treated as observable lower bounds.
- Charges are not reimbursement or revenue.
- Estimated costs are not audited hospital expenses.
- Visible cells are subject to the documented suppression policy.

## Approved Metrics

{metric_table_markdown}

## Benchmark Design

{benchmark_table_markdown}

## Interpretation Limits

The peer benchmark controls only for the fields explicitly included in its peer
definition. It does not eliminate all case-mix or hospital-structure differences.

Positive excess LOS days identify observed variation relative to the benchmark.
They do not prove inefficiency, preventability, poor quality, or causal hospital
performance.

The initial peer benchmark is distinct from the expected-LOS model that will be
developed and validated later.
"""

METRIC_DOC_PATH = DOCS_DIR / "metric_catalog.md"

METRIC_DOC_PATH.write_text(
    metric_document,
    encoding="utf-8",
)

assert METRIC_DOC_PATH.exists()
assert METRIC_DOC_PATH.stat().st_size > 0

print("METRIC_DOC_PATH detected successfully.")
print("METRIC_DOC_PATH:", METRIC_DOC_PATH.name)


METRIC_DOC_PATH detected successfully.
METRIC_DOC_PATH: metric_catalog.md


## 19. Notebook 02 Summary

## Work Completed

- Loaded and validated the outputs from `01_data_audit.ipynb`.
- Confirmed availability of benchmark-critical source fields.
- Defined analytical grain and aggregation standards.
- Created machine-readable base, LOS, financial, admission-context, case-mix,
  and peer-benchmark metrics.
- Defined denominator and invalid-record rules.
- Reviewed and documented categorical mappings.
- Defined leave-one-facility-out peer benchmarking.
- Separated descriptive peer expectations from future model predictions.
- Defined conservative small-cell suppression rules.
- Validated and exported the metric specifications.
- Generated business-readable metric documentation.

## Key Design Decisions

1. **Analytical grain:** One inpatient discharge.
2. **Time grain:** Annual for the initial dataset.
3. **LOS top-coding:** 120+ is treated as an observable lower bound.
4. **Financial aggregation:** Ratio-of-sums is used where appropriate.
5. **Peer definition:** APR-DRG and severity, with APR-DRG fallback.
6. **Peer independence:** Each facility is excluded from its own benchmark.
7. **Minimum peer size:** 30 comparison discharges.
8. **Suppression:** Visible counts from 1 through 10 are suppressed.
9. **Expected LOS:** The initial expected value is a descriptive peer baseline,
   not a machine-learning prediction.

## Important Limitations

- Discharge counts are not unique-patient counts.
- Monthly analysis is unsupported.
- True occupancy cannot be calculated without staffed-bed data.
- Readmissions cannot be identified without patient linkage.
- Top-coded LOS causes totals and means to be lower-bound estimates.
- Charges do not represent reimbursement or revenue.
- Estimated costs do not represent audited expenses.
- APR-DRG and severity do not capture every case-mix difference.
- Hospital structural characteristics are not yet incorporated.
- Peer differences do not prove inefficiency or preventability.
- No causal conclusions are supported.
- Peer-median benchmarks are retained as deferred contextual measures.
  Their aggregation across multiple APR-DRG peer groups must be defined
  before implementation.